In [ ]:
"""
Simple Hybrid Ball Tracker - YOLO Primary, Frame-Diff Backup
- YOLO tracks ball normally
- When YOLO fails, frame-diff takes over using last known position and size
- Follows the ball blob frame by frame until YOLO picks it up again
- NO fancy stuff - just reliable tracking
"""

import os
import cv2
import numpy as np
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

# ======================== CONFIG ========================
from pathlib import Path
# Attempt to resolve paths cross-platform and support common locations
ROOT = Path(__file__).parent if '__file__' in globals() else Path.cwd()
# Candidate video locations (will pick the first existing)
VIDEO_CANDIDATES = [
    ROOT / 'workspace' / 'input.MOV',
    ROOT / 'input.MOV',
    Path('input.MOV'),
    ROOT / 'workspace' / 'input.mp4',
    ROOT / 'input.mp4'
]
VIDEO_PATH = next((str(p) for p in VIDEO_CANDIDATES if p.exists()), str(VIDEO_CANDIDATES[0]))
# Output path (ensure folder exists)
OUTPUT_PATH = str(ROOT / 'workspace' / 'output' / 'trajectory_ball_tracking.mp4')
# Candidate YOLO model locations
YOLO_CANDIDATES = [
    ROOT / 'workspace' / 'yolov8x.pt',
    ROOT / 'yolov8x.pt',
    Path('yolov8x.pt')
]
YOLO_MODEL_PATH = next((str(p) for p in YOLO_CANDIDATES if p.exists()), str(YOLO_CANDIDATES[0]))

# YOLO Settings
CONFIDENCE_THRESHOLD = 0.25

# Frame Diff Settings
DIFF_THRESHOLD = 15
BLUR_KERNEL = (5, 5)
MIN_BALL_AREA = 20
MAX_BALL_AREA = 8000

# Tracking Settings
SEARCH_RADIUS_MULTIPLIER = 3.0  # How far from last position to look
SIZE_TOLERANCE = 0.4  # How much ball size can change (±40%)
MAX_MOVEMENT_PER_FRAME = 50  # Max pixels ball can move in one frame
# ========================================================


class SimpleBallTracker:
    def __init__(self):
        print("🎾 Initializing Simple Hybrid Ball Tracker...")
        self.model = YOLO(YOLO_MODEL_PATH)
        
        # Current tracking state
        self.last_position = None
        self.last_size = 15
        self.current_mode = "SEARCHING"
        self.confidence = 0.0
        
        print("   ✓ Ready")

    def find_ball_yolo(self, frame):
        """Find ball using YOLO"""
        results = self.model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
        
        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                cls_idx = int(box.cls[0])
                class_name = str(self.model.names.get(cls_idx, "")).lower().strip()
                
                if class_name in {"sports ball", "ball"}:
                    conf = float(box.conf[0])
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    cx = int((x1 + x2) / 2)
                    cy = int((y1 + y2) / 2)
                    radius = int(min(x2 - x1, y2 - y1) / 2)
                    
                    return {
                        'position': (cx, cy),
                        'size': max(3, radius),
                        'confidence': conf
                    }
        
        return None

    def find_ball_framediff(self, prev_frame, curr_frame):
        """Find ball using frame difference, looking near last known position"""
        if self.last_position is None:
            return None
        
        # Convert to grayscale and blur
        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
        curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
        prev_gray = cv2.GaussianBlur(prev_gray, BLUR_KERNEL, 0)
        curr_gray = cv2.GaussianBlur(curr_gray, BLUR_KERNEL, 0)
        
        # Get frame difference
        diff = cv2.absdiff(curr_gray, prev_gray)
        _, thresh = cv2.threshold(diff, DIFF_THRESHOLD, 255, cv2.THRESH_BINARY)
        
        # Clean up noise
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
        
        # Find contours
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # Look for blob near last position with similar size
        last_x, last_y = self.last_position
        search_radius = self.last_size * SEARCH_RADIUS_MULTIPLIER
        min_size = int(self.last_size * (1 - SIZE_TOLERANCE))
        max_size = int(self.last_size * (1 + SIZE_TOLERANCE))
        
        best_blob = None
        best_score = 0
        
        for contour in contours:
            area = cv2.contourArea(contour)
            
            # Size check
            estimated_radius = int(np.sqrt(area / np.pi))
            if estimated_radius < min_size or estimated_radius > max_size:
                continue
            
            if area < MIN_BALL_AREA or area > MAX_BALL_AREA:
                continue
            
            # Get center
            M = cv2.moments(contour)
            if M["m00"] == 0:
                continue
            
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            
            # Distance from last position
            distance = np.sqrt((cx - last_x)**2 + (cy - last_y)**2)
            
            # Skip if too far
            if distance > search_radius:
                continue
            
            # Skip if moved too much (unrealistic)
            if distance > MAX_MOVEMENT_PER_FRAME:
                continue
            
            # Score based on distance (closer = better) and size similarity
            size_diff = abs(estimated_radius - self.last_size)
            size_score = max(0, 1 - (size_diff / self.last_size))
            distance_score = max(0, 1 - (distance / search_radius))
            
            total_score = distance_score * 0.6 + size_score * 0.4
            
            if total_score > best_score:
                best_score = total_score
                best_blob = {
                    'position': (cx, cy),
                    'size': estimated_radius,
                    'confidence': min(0.8, total_score),
                    'distance_moved': distance
                }
        
        return best_blob if best_score > 0.3 else None

    def track_ball(self, frame, prev_frame=None):
        """Main tracking function"""
        # Try YOLO first
        yolo_result = self.find_ball_yolo(frame)
        
        if yolo_result is not None:
            # YOLO found the ball
            self.last_position = yolo_result['position']
            self.last_size = yolo_result['size']
            self.confidence = yolo_result['confidence']
            self.current_mode = "YOLO"
            
            return {
                'found': True,
                'position': yolo_result['position'],
                'size': yolo_result['size'],
                'confidence': yolo_result['confidence'],
                'mode': 'YOLO'
            }
        
        # YOLO failed - try frame diff if we have previous tracking
        elif prev_frame is not None and self.last_position is not None:
            framediff_result = self.find_ball_framediff(prev_frame, frame)
            
            if framediff_result is not None:
                # Frame diff found the ball
                self.last_position = framediff_result['position']
                self.last_size = framediff_result['size']
                self.confidence = framediff_result['confidence']
                self.current_mode = "FRAME_DIFF"
                
                return {
                    'found': True,
                    'position': framediff_result['position'],
                    'size': framediff_result['size'],
                    'confidence': framediff_result['confidence'],
                    'mode': 'FRAME_DIFF',
                    'movement': framediff_result['distance_moved']
                }
            
            # Neither method found ball but keep last known position briefly
            else:
                self.confidence *= 0.7  # Decay confidence
                self.current_mode = "LOST"
                
                if self.confidence > 0.1:  # Still have some confidence
                    return {
                        'found': True,
                        'position': self.last_position,
                        'size': self.last_size,
                        'confidence': self.confidence,
                        'mode': 'PREDICTED'
                    }
                else:
                    # Completely lost
                    return {
                        'found': False,
                        'position': None,
                        'size': None,
                        'confidence': 0.0,
                        'mode': 'LOST'
                    }
        
        # No previous frame or position
        else:
            self.current_mode = "SEARCHING"
            return {
                'found': False,
                'position': None,
                'size': None,
                'confidence': 0.0,
                'mode': 'SEARCHING'
            }

    def draw_result(self, frame, result):
        """Draw tracking result on frame"""
        output = frame.copy()
        
        if result['found'] and result['position'] is not None:
            cx, cy = result['position']
            radius = result['size']
            
            # Choose color based on tracking mode
            if result['mode'] == 'YOLO':
                color = (0, 255, 0)  # Green
                thickness = 2
            elif result['mode'] == 'FRAME_DIFF':
                color = (0, 255, 255)  # Yellow
                thickness = 2
            elif result['mode'] == 'PREDICTED':
                color = (0, 165, 255)  # Orange
                thickness = 1
            else:
                color = (0, 0, 255)  # Red
                thickness = 1
            
            # Draw circle around ball
            cv2.circle(output, (cx, cy), radius, color, thickness)
            cv2.circle(output, (cx, cy), 2, color, -1)
            
            # Draw confidence bar
            if result['confidence'] > 0:
                conf_width = int(80 * result['confidence'])
                cv2.rectangle(output, (cx - 40, cy - radius - 25), 
                             (cx - 40 + conf_width, cy - radius - 20), color, -1)
                cv2.rectangle(output, (cx - 40, cy - radius - 25), 
                             (cx + 40, cy - radius - 20), (255, 255, 255), 1)
        
        # Status text
        mode_colors = {
            'YOLO': (0, 255, 0),
            'FRAME_DIFF': (0, 255, 255),
            'PREDICTED': (0, 165, 255),
            'LOST': (0, 0, 255),
            'SEARCHING': (128, 128, 128)
        }
        
        mode_color = mode_colors.get(result['mode'], (255, 255, 255))
        cv2.putText(output, f"Mode: {result['mode']}", (20, 30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, mode_color, 2)
        
        if result['found'] and result['position']:
            cv2.putText(output, f"Position: ({result['position'][0]}, {result['position'][1]})", 
                       (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            cv2.putText(output, f"Size: {result['size']*2}px", 
                       (20, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        cv2.putText(output, f"Confidence: {result['confidence']:.2f}", 
                   (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        return output


def main():
    print("=" * 60)
    print("🎾 SIMPLE HYBRID BALL TRACKER")
    print("   YOLO primary → Frame-diff backup → Seamless handoff")
    print("=" * 60)

    # Check files exist
    if not os.path.exists(VIDEO_PATH):
        print(f"❌ Video not found: {VIDEO_PATH}")
        return
    if not os.path.exists(YOLO_MODEL_PATH):
        print(f"❌ YOLO model not found: {YOLO_MODEL_PATH}")
        return

    # Setup
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    tracker = SimpleBallTracker()

    # Open video
    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"📹 Video: {width}x{height} @ {fps:.1f}fps, {total_frames} frames")

    # Setup output
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

    print("\n🎬 Processing...")
    
    frame_count = 0
    prev_frame = None
    
    # Stats
    yolo_frames = 0
    framediff_frames = 0
    predicted_frames = 0
    lost_frames = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Track the ball
        result = tracker.track_ball(frame, prev_frame)
        
        # Draw and save result
        output_frame = tracker.draw_result(frame, result)
        out.write(output_frame)
        
        # Update stats
        if result['mode'] == 'YOLO':
            yolo_frames += 1
        elif result['mode'] == 'FRAME_DIFF':
            framediff_frames += 1
        elif result['mode'] == 'PREDICTED':
            predicted_frames += 1
        else:
            lost_frames += 1
        
        # Progress update
        if frame_count % 60 == 0:  # Every 2 seconds at 30fps
            progress = (frame_count / total_frames) * 100 if total_frames > 0 else 0
            print(f"   Frame {frame_count}/{total_frames} ({progress:.1f}%) - {result['mode']}")
        
        prev_frame = frame
        frame_count += 1

    # Cleanup
    cap.release()
    out.release()
    cv2.destroyAllWindows()

    # Final stats
    print("\n📊 TRACKING RESULTS:")
    print(f"   YOLO tracking: {yolo_frames} frames ({yolo_frames/frame_count*100:.1f}%)")
    print(f"   Frame-diff tracking: {framediff_frames} frames ({framediff_frames/frame_count*100:.1f}%)")
    print(f"   Predicted tracking: {predicted_frames} frames ({predicted_frames/frame_count*100:.1f}%)")
    print(f"   Lost tracking: {lost_frames} frames ({lost_frames/frame_count*100:.1f}%)")
    
    successful_tracking = yolo_frames + framediff_frames + predicted_frames
    print(f"\n🎯 Total successful tracking: {successful_tracking}/{frame_count} ({successful_tracking/frame_count*100:.1f}%)")

    print(f"\n✅ Complete! Saved to: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()